[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/04_phase2_stage3_multiasset_reference.ipynb)

# 04 · Stage 3: multi-asset ground truth

**Phase:** Phase 2: multi-asset

**Goal:** Exact d-asset common-jump density, Monte Carlo reference, and a 2-D FD solver that shows how grid cost explodes with dimension (motivation for RQ4).

**Outputs:** `figures/stage3_reference_nd/`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    !git -C "{ROOT}" pull -q --ff-only
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## Plan for this stage
**Already available in the library:** `MultiMertonParams`, `multi_merton_logpdf`, `simulate_multi_merton`, and `portfolio(w)`, the exact 1-D law of a linear portfolio, which is free ground truth.

**To build:** `scripts/stage3_reference_nd.py` with a 2-D FD solver (ADI/Douglas plus a 1-D common-jump convolution along β).

**Planned figures:**
- fig01 2-D density contours, exact vs FD
- fig02 FD time/memory vs grid size, and n^d growth
- fig03 MC tail error vs d at fixed N
- fig04 exact linear-portfolio ES vs MC for several w

## Preview with the current library

In [ ]:
# Live preview with what exists today: exact 2-asset density vs Monte Carlo
from pinn_tailrisk.jumpdiff import DAY, MultiMertonParams, multi_merton_logpdf, simulate_multi_merton, merton_var_es
from pinn_tailrisk.risk import var_es_from_samples
mp = MultiMertonParams.equicorrelated(2, rho=0.5, beta=[1.0, 1.2])
T = 10 * DAY
g = np.linspace(-0.45, 0.25, 220); X, Y = np.meshgrid(g, g)
L = multi_merton_logpdf(np.stack([X, Y], -1), T, mp) / np.log(10)
fig, ax = plt.subplots(figsize=(5, 4.3))
cs = ax.contourf(X * 100, Y * 100, L, levels=np.arange(-6, 3, 0.5), cmap="seq_blue", extend="min")
ax.set(xlabel="asset 1 log return (%)", ylabel="asset 2 log return (%)", title="Exact 10-day log₁₀ density (common jumps)")
ax.grid(False); fig.colorbar(cs, ax=ax).outline.set_visible(False); plt.show()

w = np.array([0.6, 0.4])
xs = simulate_multi_merton(1_000_000, T, mp, np.random.default_rng(0)) @ w
print("ES99 exact (1-D projection):", merton_var_es(0.99, T, mp.portfolio(w))[1], " MC:", var_es_from_samples(xs, 0.99)[1])

## Run the stage
_Placeholder: the stage script is not written yet._

In [ ]:
# !python scripts/stage3_multiasset_reference.py
# show_figures("...")